## Libraries

In [32]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn import clone
from sklearn.feature_selection import RFE
from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, median_absolute_error, r2_score
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import ParameterGrid

## Import

In [33]:
X = pd.read_csv('../new_datasets/X_trainval_preprocessed.csv')
y = pd.read_csv('../new_datasets\y_trainval.csv').values.ravel()

<>:2: SyntaxWarning: invalid escape sequence '\y'
<>:2: SyntaxWarning: invalid escape sequence '\y'
C:\Users\User\AppData\Local\Temp\ipykernel_14892\893551437.py:2: SyntaxWarning: invalid escape sequence '\y'
  y = pd.read_csv('../new_datasets\y_trainval.csv').values.ravel()


In [34]:
X.head()

,Brand,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage,mileage_per_year,tax_engineSize,age_mileage,age
0,VW,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,4.000000,0.0,3157.888889,NaN,255789.0,9.0
1,Toyota,Manual,4589.0,Petrol,145.0,47.900000,1.5,1.000000,0.0,764.833333,217.5,27534.0,6.0
2,Audi,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,4.000000,0.0,604.000000,217.5,21744.0,6.0
3,Ford,Manual,9102.0,Petrol,145.0,65.700000,1.0,2.340306,0.0,1300.285714,145.0,63714.0,7.0
4,BMW,Manual,1000.0,Petrol,145.0,42.800000,1.5,3.000000,0.0,166.666667,217.5,6000.0,6.0


## Functions

In [35]:
def select_rfe(X, y, names, n_features=30):
    # Proteção: Se tivermos menos colunas que o n_features desejado, ajustamos
    n_features = min(n_features, X.shape[1])
    
    rfe = RFE(estimator=Ridge(alpha=1.0), n_features_to_select=n_features, step=0.1)
    rfe.fit(X, y)
    
    # PROTEÇÃO CRÍTICA: Iteramos apenas até ao limite seguro
    limit = min(len(names), len(rfe.support_))
    
    return [names[i] for i in range(limit) if rfe.support_[i]]

In [36]:
# Função 2: Ridge (Ultra-Robust)
def select_ridge(X, y, names, alpha=1.0):
    ridge = Ridge(alpha=alpha)
    ridge.fit(X, y)
    
    # PROTEÇÃO CRÍTICA: O limite é o menor entre nomes e coeficientes
    limit = min(len(names), len(ridge.coef_))
    
    return [names[i] for i in range(limit) if abs(ridge.coef_[i]) > 1e-4]

In [37]:
# Função 3: Lasso (Ultra-Robust)
def select_lasso(X, y, names, alpha=0.01):
    lasso = Lasso(alpha=alpha)
    lasso.fit(X, y)
    
    # PROTEÇÃO CRÍTICA
    limit = min(len(names), len(lasso.coef_))
    
    return [names[i] for i in range(limit) if lasso.coef_[i] != 0]

In [38]:
def consensus_features(X, y, names):
    # Garante que 'names' nunca é maior que as colunas do X (corta o excesso)
    min_len = X.shape[1]
    safe_names = names[:min_len]
    
    # Agora passamos os safe_names e as funções internas já tratam se safe_names for curto demais
    f_rfe   = set(select_rfe(X, y, safe_names, n_features=30))
    f_ridge = set(select_ridge(X, y, safe_names))
    f_lasso = set(select_lasso(X, y, safe_names))

    return sorted(list(f_rfe & f_ridge & f_lasso))

## Models

In [39]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1),
    'Random Forest': RandomForestRegressor(n_estimators= 100, random_state=42, max_depth = 20, n_jobs=-1),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300,learning_rate=0.1,max_depth=5,random_state=42),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(100,50), max_iter=500, random_state=42)
}

## Metrics

In [40]:
metrics = {
    'R2': r2_score,
    'MAE': mean_absolute_error,
    'MSE': mean_squared_error,
    'MAPE': mean_absolute_percentage_error,
    'MedAE': median_absolute_error
}

## Cross-Validation

In [41]:
#Using the Standard Cross-Validation to evaluate the model
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [42]:
results = {name: {m: [] for m in metrics} for name in models}

In [43]:
fold = 1
for train_index, val_index in cv.split(X,y):
    print(f"--> A processar Fold {fold}/5...")
# 1- Data Split
    X_train_fold = X.iloc[train_index].copy()
    X_val_fold = X.iloc[val_index].copy()

    y_train_fold = y[train_index]
    y_val_fold = y[val_index]
    

# 2- Preprocessing
    # Seperate the numeric columns
    numeric_cols = X_train_fold.select_dtypes(include=np.number).columns

    # Impute the null values
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train_fold[numeric_cols])  

    # Apply the transformation
    X_train_fold[numeric_cols] = imputer.transform(X_train_fold[numeric_cols])
    X_val_fold[numeric_cols] = imputer.transform(X_val_fold[numeric_cols])

    #Encoding
    categorical_cols = X_train_fold.select_dtypes(include=['object', 'bool']).columns

    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_fold[categorical_cols])

    X_train_encoded = encoder.transform(X_train_fold[categorical_cols])
    X_val_encoded = encoder.transform(X_val_fold[categorical_cols])

    X_train_final = np.hstack([X_train_fold[numeric_cols].values, X_train_encoded])
    X_val_final = np.hstack([X_val_fold[numeric_cols].values, X_val_encoded])

    encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
    all_feature_names = list(numeric_cols) + list(encoded_feature_names)

# 3- Scaling
    scaler = MinMaxScaler()
    scaler.fit(X_train_final) # Fit on TRAIN
    
    X_train_scaled = scaler.transform(X_train_final)
    X_val_scaled = scaler.transform(X_val_final)


# 4 - Feature Selection
    # Chama a função de consenso passando os nomes das colunas
    selected = consensus_features(X_train_scaled, y_train_fold, all_feature_names)

    # Encontrar os índices das colunas selecionadas
    sel_idx = [all_feature_names.index(f) for f in selected]

    # Reduzir as matrizes de treino e validação
    X_train_final = X_train_scaled[:, sel_idx]
    X_val_final   = X_val_scaled[:, sel_idx]
    
# 5 - Training and Evaluation
    for name, model in models.items():
        m = clone(model)
        m.fit(X_train_final, y_train_fold)
        pred = m.predict(X_val_final)
        
        for m_name, func in metrics.items():
            score = func(y_val_fold, pred)
            results[name][m_name].append(score)
            
    fold += 1



--> A processar Fold 1/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.968e+10, tolerance: 5.647e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


--> A processar Fold 2/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.242e+10, tolerance: 5.650e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


--> A processar Fold 3/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.661e+10, tolerance: 5.623e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


--> A processar Fold 4/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.459e+10, tolerance: 5.601e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


--> A processar Fold 5/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.698e+10, tolerance: 5.609e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


## Models 

In [44]:
# --- RESULTS TABLE ---
# Convert the dictionary of lists into a DataFrame
results_df = pd.DataFrame(results).T

# Calculate the mean of the 5 folds for each metric
# (This gives you the final "Score" for each model)
metrics_table = results_df.map(lambda x: np.mean(x))

# Display the table sorted by R2 (or MAE)
print("\n--- Model Performance (Cross-Validation Mean) ---")
display(metrics_table.round(4).sort_values(by='R2', ascending=False))


--- Model Performance (Cross-Validation Mean) ---


,R2,MAE,MSE,MAPE,MedAE
Random Forest,0.9219,1567.1668,7.407017e+06,0.0980,942.0850
Gradient Boosting,0.9217,1720.0166,7.425532e+06,0.1077,1137.1972
Decision Tree,0.8670,2016.9531,1.261327e+07,0.1260,1134.6000
Neural Network,0.8657,2175.5009,1.273550e+07,0.1354,1427.6403
Linear Regression,0.7475,3039.5434,2.395199e+07,0.2134,2103.8613
Ridge,0.7464,3034.9665,2.405815e+07,0.2125,2080.7911


In [49]:
# --- GRID SEARCH "DEEP DIVE" PARA RANDOM FOREST ---
# Total de Combinações: ~300 (x 5 Folds = 1500 treinos)
# Estimativa de Tempo: 20 a 40 minutos (dependendo do CPU)

rf_grid = {
    'model_name': ['Random Forest'],
    'model': [RandomForestRegressor(random_state=42, n_jobs=-1)],
    'params': {
        # 1. Quantidade de Árvores:
        # Testamos valores mais altos para estabilizar a previsão
        'n_estimators': [300, 500, 800],

        # 2. Profundidade da Árvore:
        # None = cresce até ao fim (pode overfitar). 20 e 30 controlam.
        'max_depth': [None, 20, 30],

        # 3. Robustez (Features):
        # 'sqrt' vê menos colunas (mais rápido, menos overfit).
        # 1.0 vê todas as colunas (melhor se tiveres poucas features boas).
        'max_features': ['sqrt', 1.0],

        # 4. Controlo de Folhas (O segredo do MAE):
        # min_samples_split: Mínimo de carros para dividir um nó (2, 5, 10)
        # min_samples_leaf: Mínimo de carros numa folha final (1, 2, 4)
        # Valores mais altos aqui evitam que o modelo decore carros únicos.
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],

        # 5. Bootstrap:
        # True = Usa amostragem aleatória (padrão, reduz variância).
        # False = Usa o dataset todo para cada árvore (pode ser brutalmente preciso mas perigoso).
        'bootstrap': [True]
    }
}

grids_to_test = [rf_grid]
grid_results = {}


In [50]:
print("A iniciar Grid Search Manual...")

# 2. LOOP DE VALIDAÇÃO (O mesmo de sempre)
fold = 1
for train_index, val_index in cv.split(X, y):
    print(f"--> A processar Fold {fold}/5...")
    
    # --- A) DATA SPLIT ---
    X_train_fold = X.iloc[train_index].copy()
    X_val_fold = X.iloc[val_index].copy()
    y_train_fold = y[train_index]
    y_val_fold = y[val_index]

    # --- B) PREPROCESSING ---
    # 1. Impute
    numeric_cols = X_train_fold.select_dtypes(include=np.number).columns
    imputer = SimpleImputer(strategy='median')
    imputer.fit(X_train_fold[numeric_cols])
    X_train_fold[numeric_cols] = imputer.transform(X_train_fold[numeric_cols])
    X_val_fold[numeric_cols] = imputer.transform(X_val_fold[numeric_cols])

    # 2. Encode
    categorical_cols = X_train_fold.select_dtypes(include=['object', 'bool']).columns
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_fold[categorical_cols])
    X_train_encoded = encoder.transform(X_train_fold[categorical_cols])
    X_val_encoded = encoder.transform(X_val_fold[categorical_cols])

    X_train_processed = np.hstack([X_train_fold[numeric_cols].values, X_train_encoded])
    X_val_processed = np.hstack([X_val_fold[numeric_cols].values, X_val_encoded])

    # 3. Scale
    scaler = MinMaxScaler()
    scaler.fit(X_train_processed)
    X_train_scaled = scaler.transform(X_train_processed)
    X_val_scaled = scaler.transform(X_val_processed)

    # 4. Feature Selection (SelectFromModel - RF)
    # Chama a função de consenso passando os nomes das colunas
    selected = consensus_features(X_train_scaled, y_train_fold, all_feature_names)

    # Encontrar os índices das colunas selecionadas
    sel_idx = [all_feature_names.index(f) for f in selected]

    # Reduzir as matrizes de treino e validação
    X_train_final = X_train_scaled[:, sel_idx]
    X_val_final   = X_val_scaled[:, sel_idx]
    
    # --- C) GRID SEARCH INTERNO ---
    for entry in grids_to_test:
        base_model = entry['model'][0]
        base_name = entry['model_name'][0]
        
        # CORREÇÃO AQUI: Usar ParameterGrid para gerar as combinações
        param_grid = list(ParameterGrid(entry['params']))
        
        for params in param_grid:
            # Criar nome único para esta configuração
            combo_name = f"{base_name} | {str(params)}"
            
            # Inicializar se for novo
            if combo_name not in grid_results:
                grid_results[combo_name] = {m: [] for m in metrics}
            
            # Clonar e configurar
            m = clone(base_model)
            m.set_params(**params)
            
            m.fit(X_train_final, y_train_fold)
            pred = m.predict(X_val_final)
            
            # Guardar Scores
            for m_name, func in metrics.items():
                score = func(y_val_fold, pred)
                grid_results[combo_name][m_name].append(score)
    
    fold += 1

# MOSTRAR O VENCEDOR
print("\n--- Tuning Concluído! ---")
results_df = pd.DataFrame(grid_results).T
final_metrics = results_df.map(lambda x: np.mean(x))

print("\n--- TOP 5 Melhores Configurações (Ordenado por MAE) ---")
display(final_metrics.round(4).sort_values(by='MAE', ascending=True).head(5))

A iniciar Grid Search Manual...
--> A processar Fold 1/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.968e+10, tolerance: 5.647e+08
  model = cd_fast.enet_coordinate_descent(


--> A processar Fold 2/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.242e+10, tolerance: 5.650e+08
  model = cd_fast.enet_coordinate_descent(


--> A processar Fold 3/5...


...

c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.459e+10, tolerance: 5.601e+08
  model = cd_fast.enet_coordinate_descent(


--> A processar Fold 5/5...


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.698e+10, tolerance: 5.609e+08
  model = cd_fast.enet_coordinate_descent(



--- Tuning Concluído! ---

--- TOP 5 Melhores Configurações (Ordenado por MAE) ---


,R2,MAE,MSE,MAPE,MedAE
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800}",0.9224,1559.5422,7.364527e+06,0.0975,939.7381
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 500}",0.9223,1559.8763,7.371077e+06,0.0975,940.7579
"Random Forest | {'bootstrap': True, 'max_depth': 30, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800}",0.9221,1560.1234,7.387091e+06,0.0976,939.1434
"Random Forest | {'bootstrap': True, 'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 800}",0.9222,1560.2082,7.384399e+06,0.0976,938.3819
"Random Forest | {'bootstrap': True, 'max_depth': 20, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 800}",0.9227,1560.4938,7.328956e+06,0.0976,937.5674


## Best Model

In [57]:
# Célula [54] - Best Model (Mantida como está)
final_model = RandomForestRegressor(
    n_estimators=800, 
    max_depth=20, 
    max_features=1.0,  
    min_samples_split=5,
    min_samples_leaf=1,
    bootstrap=True,
    random_state=42
)

In [58]:
# 2. Carregar e Preparar Dados (Pipeline de Transformação e Feature Selection)

# Carregar o teste original 
X_test_kaggle = pd.read_csv('../new_datasets/X_test_preprocessed.csv')
X_full = X.copy()
y_full = y.copy()

# --- A) Impute ---
numeric_cols = X_full.select_dtypes(include=np.number).columns
imputer = SimpleImputer(strategy='median')
imputer.fit(X_full[numeric_cols])

X_full[numeric_cols] = imputer.transform(X_full[numeric_cols])
X_test_kaggle[numeric_cols] = imputer.transform(X_test_kaggle[numeric_cols])

# --- B) Encode ---
categorical_cols = X_full.select_dtypes(include=['object', 'bool']).columns
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
encoder.fit(X_full[categorical_cols])

X_full_enc = encoder.transform(X_full[categorical_cols])
X_test_enc = encoder.transform(X_test_kaggle[categorical_cols])

# Juntar Arrays e obter nomes de todas as features (numéricas + dummy)
X_full_processed = np.hstack([X_full[numeric_cols].values, X_full_enc])
X_test_processed = np.hstack([X_test_kaggle[numeric_cols].values, X_test_enc])
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)
all_feature_names = list(numeric_cols) + list(encoded_feature_names) # Essencial para o consenso

# --- C) Scale ---
scaler = MinMaxScaler()
scaler.fit(X_full_processed)

X_full_scaled = scaler.transform(X_full_processed)
X_test_scaled = scaler.transform(X_test_processed)

# --- D) Feature Selection (Aplicação do consenso na TOTALIDADE dos dados de treino) ---
print("A realizar Feature Selection por consenso...")
selected = consensus_features(X_full_scaled, y_full, all_feature_names)

# Encontrar os índices das colunas selecionadas nos arrays escalados
sel_idx = [all_feature_names.index(f) for f in selected]

# Reduzir as matrizes de treino e teste
X_full_final = X_full_scaled[:, sel_idx]
X_test_final = X_test_scaled[:, sel_idx]

print(f"Dados prontos! Features finais: {X_full_final.shape[1]}")
print(f"Features selecionadas: {selected}")

A realizar Feature Selection por consenso...
Dados prontos! Features finais: 27
Features selecionadas: ['Brand_BMW', 'Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Unknown', 'Brand_VW', 'age', 'age_mileage', 'engineSize', 'fuelType_Electric', 'fuelType_Hybrid', 'fuelType_Other', 'fuelType_Petrol', 'fuelType_Unknown', 'mileage', 'mileage_per_year', 'mpg', 'previousOwners', 'tax', 'tax_engineSize', 'transmission_Manual', 'transmission_Other', 'transmission_Semi-Auto', 'transmission_Unknown']


c:\Users\User\anaconda3\envs\Fall2526\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.295e+10, tolerance: 7.033e+08
  model = cd_fast.enet_coordinate_descent(


## Kaggle

In [59]:
# 3. Treinar e Prever
print("A treinar modelo final...")
final_model.fit(X_full_final, y_full)

print("A gerar submissão...")
y_pred = final_model.predict(X_test_final)

# 4. Criar CSV
submission = pd.DataFrame({
    'CarID': pd.read_csv('../original_datasets/test.csv')['carID'], # Garante IDs originais
    'price': y_pred
})
submission.to_csv('kaggle_submission.csv', index=False)
print("Sucesso! 'kaggle_submission.csv' criado.")

A treinar modelo final...
A gerar submissão...
Sucesso! 'kaggle_submission.csv' criado.
